# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [18]:
! pip install mlflow==2.12.2

Looking in indexes: https://package-mirror.liara.ir/repository/pypi/simple



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

In [20]:
MLFLOW_TRACKING_URI = "http://185.50.38.163:33014"

# TODO: replace these with your assigned MLflow credentials.
MLFLOW_USERNAME = "student_ayda_hafezian"
MLFLOW_PASSWORD = "zK4uJiJvGPLZFx5WDRc"
EXPERIMENT_NAME = "qbc12_hw02_student_ayda_hafezian"

if MLFLOW_USERNAME == "student_your_username" or MLFLOW_PASSWORD == "your_mlflow_password":
    raise ValueError("Replace the MLflow placeholders with your real credentials.")

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)

MLflow tracking URI: http://185.50.38.163:33014
Experiment: qbc12_hw02_student_ayda_hafezian
Experiment ID: 8


## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [21]:
from pathlib import Path
print(list(Path(".").rglob("*features*")))


[WindowsPath('data/features'), WindowsPath('data/features/listing_availability_features_v1_student.csv'), WindowsPath('data/features/listing_availability_features_v1_student.parquet'), WindowsPath('data/features/listing_availability_features_v1_student_metadata.json'), WindowsPath('data/features/listing_availability_features_v1_student_validation_report.json')]


In [22]:
DATASET_VERSION = "v1_student"

FEATURE_DIR = Path("data/features")

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path     = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

# TODO: load the dataset.
# Prefer Parquet if it exists, otherwise use CSV.
if parquet_path.exists():
    feature_df = pd.read_parquet(parquet_path)
elif csv_path.exists():
    feature_df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError("Dataset not found in data/features/")

# TODO: load metadata if metadata_path exists.
metadata = {}
if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

print(feature_df.shape)
feature_df.head()

(10480, 32)


,listing_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms,minimum_nights,maximum_nights,instant_bookable,...,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version
0,27886,Private room,Private room in houseboat,2,1.0,1.0,1.5,3,356,False,...,0,0.000000,3.0,30.0,30,0,0.0,1,2026-08-11,v1_student
1,28871,Private room,Private room in rental unit,2,1.0,1.0,1.0,2,730,False,...,14,0.466667,2.0,730.0,30,21,0.7,0,2026-08-11,v1_student
2,29051,Private room,Private room in condo,2,1.0,1.0,1.0,2,730,False,...,16,0.533333,2.0,730.0,30,0,0.0,1,2026-08-11,v1_student
3,44391,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,3,730,False,...,0,0.000000,3.0,730.0,30,0,0.0,1,2026-08-11,v1_student
4,48373,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,3,1125,False,...,0,0.000000,3.0,1125.0,30,0,0.0,1,2026-08-11,v1_student


## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [23]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

# TODO: check that TARGET_COL exists.
if TARGET_COL not in feature_df.columns:
    raise ValueError("Target column not found")

# TODO: create y.
y = feature_df[TARGET_COL].copy()

# TODO: create clean feature list by excluding FORBIDDEN_MODEL_COLUMNS.
clean_feature_cols = [c for c in feature_df.columns if c not in FORBIDDEN_MODEL_COLUMNS]

# TODO: create X_clean.
X_clean = feature_df[clean_feature_cols].copy()

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())

print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)

Target distribution:
high_demand_proxy
0    0.237214
1    0.762786
Name: proportion, dtype: Float64
Clean feature count: 25
['room_type', 'property_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms', 'minimum_nights', 'maximum_nights', 'instant_bookable', 'is_superhost', 'host_listing_count', 'neighbourhood_name', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff', 'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d', 'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d', 'available_days_last_30d', 'available_rate_last_30d', 'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d']


## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [24]:
LEAKAGE_COLUMN = "future_available_rate_30d"

# TODO: create leaky_feature_cols.
# It should include the clean features plus LEAKAGE_COLUMN.
# It must still exclude the target itself.

if LEAKAGE_COLUMN not in feature_df.columns:
    raise ValueError("Leakage column missing.")

leaky_feature_cols = clean_feature_cols + [LEAKAGE_COLUMN]
X_leaky = feature_df[leaky_feature_cols].copy()

print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)

Leaky feature count: 26
Leakage column included: True


## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [25]:
# TODO: split X_clean and y.
# Use test_size=0.20, random_state=42, stratify=y.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train shape: (8384, 25)
Test shape: (2096, 25)
Train target rate: 0.7627624045801527
Test target rate: 0.762881679389313


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [26]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# TODO: identify numeric_cols and categorical_cols from X_clean.
# Hint: numeric columns usually have dtype int/float.
# Everything else can be treated as categorical.

numeric_cols = X_clean.select_dtypes(include=["int64","float64","int32","float32","bool"]).columns.tolist()
categorical_cols = [c for c in X_clean.columns if c not in numeric_cols]


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 22
Categorical columns: 3


## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [29]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    # TODO:
    # 1. get positive scores
    # 2. convert scores to predictions using threshold
    # 3. calculate accuracy, precision, recall, f1, roc_auc
    # 4. return metrics dict, y_pred, y_score

    y_score = get_positive_scores(model, X_test)
    y_pred  = (y_score >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score)
    }

    return metrics, y_pred, y_score

## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [33]:
ARTIFACT_DIR = Path("outputs/mlflow_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    # TODO:
    # 1. create a run-specific artifact folder
    # 2. save confusion_matrix.png
    # 3. save classification_report.json
    # 4. save feature_columns.json
    # 5. save dataset_metadata_snapshot.json

    folder = ARTIFACT_DIR / run_name.replace(" ","_")
    folder.mkdir(parents=True, exist_ok=True)

    # confusion matrix
    fig, ax = plt.subplots(figsize=(4,4))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax, cmap="Blues")
    fig.savefig(folder / "confusion_matrix.png", dpi=120)
    plt.close(fig)

    # classification report
    with open(folder/"classification_report.json","w") as f:
        json.dump(classification_report(y_true, y_pred, output_dict=True), f, indent=2)

    # feature cols
    with open(folder/"feature_columns.json","w") as f:
        json.dump(feature_cols, f, indent=2)

    # metadata snapshot
    snapshot = dict(metadata)
    snapshot["dataset_version"] = DATASET_VERSION
    with open(folder/"dataset_metadata_snapshot.json","w") as f:
        json.dump(snapshot, f, indent=2)

    return folder

## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [36]:
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    # TODO: implement this function.
    # Required MLflow calls:
    # - mlflow.start_run(run_name=run_name)
    # - mlflow.log_params(...)
    # - mlflow.log_metrics(...)
    # - mlflow.set_tags(...)
    # - mlflow.log_artifacts(...)
    # - mlflow.sklearn.log_model(...)
    with mlflow.start_run(run_name=run_name) as run:

        # fit
        pipeline.fit(X_train, y_train)

        # evaluate
        metrics, y_pred, y_score = evaluate_binary_classifier(
            pipeline, X_test, y_test, threshold
        )

        # log params
        mlflow.log_params(model_params)

        # log metrics
        mlflow.log_metrics(metrics)

        # tags
        mlflow.set_tags(tags)

        # artifacts
        f = save_run_artifacts(run_name, y_test, y_pred, feature_cols, metadata)
        mlflow.log_artifacts(str(f))

        # save model
        mlflow.sklearn.log_model(pipeline, "model")

        print("Run completed:", run_name)
        print("Run ID:", run.info.run_id)
        print(metrics)

        return run.info.run_id

## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [38]:
# TODO:
# 1. split X_leaky and y using the same stratified split settings
# 2. build a LogisticRegression pipeline
# 3. log the run to MLflow

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky, y, test_size=0.20, random_state=42, stratify=y
)

leaky_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

run0 = run_mlflow_experiment(
    "v0_leaky_logistic_regression",
    leaky_pipeline,
    X_train_leaky, X_test_leaky,
    y_train_leaky, y_test_leaky,
    leaky_feature_cols,
    model_params={"model":"logreg","leaky":True},
    tags={
        "leakage_status":"leaky",
        "known_defect":"uses future_available_rate_30d",
        "model_family":"logistic_regression"
    }
)

Run completed: v0_leaky_logistic_regression
Run ID: 5d70c3a28a54428bade2b0b697e1de96
{'accuracy': 0.9756679389312977, 'precision': 0.9754299754299754, 'recall': 0.9931207004377736, 'f1': 0.9841958475364115, 'roc_auc': 0.9881691650843145}


## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [40]:
# TODO: build and log dummy baseline.

dummy_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

run1 = run_mlflow_experiment(
    "v1_dummy_baseline",
    dummy_pipeline,
    X_train, X_test,
    y_train, y_test,
    clean_feature_cols,
    model_params={"model":"dummy"},
    tags={"leakage_status":"clean","model_family":"dummy"}
)

c:\Users\aidah\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\aidah\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\aidah\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Run completed: v1_dummy_baseline
Run ID: d46b3dab31af410ca1a759e10d63caba
{'accuracy': 0.762881679389313, 'precision': 0.762881679389313, 'recall': 1.0, 'f1': 0.8654939106901218, 'roc_auc': 0.5}


## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [42]:
# TODO: build and log clean LogisticRegression.

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

run2 = run_mlflow_experiment(
    "v2_clean_logistic_regression",
    lr_pipeline,
    X_train, X_test,
    y_train, y_test,
    clean_feature_cols,
    model_params={"model":"logreg","class_weight":None},
    tags={"leakage_status":"clean","model_family":"logistic_regression"}
)

Run completed: v2_clean_logistic_regression
Run ID: 77f19294590b4ee6b52e6fbb095e2d61
{'accuracy': 0.9756679389312977, 'precision': 0.9754299754299754, 'recall': 0.9931207004377736, 'f1': 0.9841958475364115, 'roc_auc': 0.9881691650843145}


## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [44]:
# TODO: build and log class-weighted LogisticRegression.

lr_balanced = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

run3 = run_mlflow_experiment(
    "v3_balanced_logistic_regression",
    lr_balanced,
    X_train, X_test,
    y_train, y_test,
    clean_feature_cols,
    model_params={"model":"logreg","class_weight":"balanced"},
    tags={"leakage_status":"clean","model_family":"logistic_regression"}
)

Run completed: v3_balanced_logistic_regression
Run ID: 6e15e7fea0844db18c3781079ccc8127
{'accuracy': 0.9770992366412213, 'precision': 0.9819763828464885, 'recall': 0.9881175734834271, 'f1': 0.9850374064837906, 'roc_auc': 0.9889329724437934}


## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [46]:
# TODO: log threshold-tuning runs.

thresholds = [0.30, 0.40, 0.50, 0.60]

runs_threshold = []
for th in thresholds:
    run_id = run_mlflow_experiment(
        f"v4_threshold_{th}",
        lr_balanced,
        X_train, X_test,
        y_train, y_test,
        clean_feature_cols,
        model_params={"model":"logreg","threshold":th},
        tags={"leakage_status":"clean","model_family":"logistic_regression"},
        threshold=th
    )
    runs_threshold.append(run_id)

Run completed: v4_threshold_0.3
Run ID: 62b2d4057a2f40d19294888b8191bc13
{'accuracy': 0.9756679389312977, 'precision': 0.9771886559802713, 'recall': 0.9912445278298937, 'f1': 0.9841664079478423, 'roc_auc': 0.9889329724437934}
Run completed: v4_threshold_0.4
Run ID: 28e3e8207ca14b78a5be4d9bbd3af30e
{'accuracy': 0.9770992366412213, 'precision': 0.9795918367346939, 'recall': 0.9906191369606003, 'f1': 0.9850746268656716, 'roc_auc': 0.9889329724437934}
Run completed: v4_threshold_0.5
Run ID: 69b74317a99d40e5bbdd75792a07801e
{'accuracy': 0.9770992366412213, 'precision': 0.9819763828464885, 'recall': 0.9881175734834271, 'f1': 0.9850374064837906, 'roc_auc': 0.9889329724437934}
Run completed: v4_threshold_0.6
Run ID: da423c75e2c94ca0b2a8b104ca1fc952
{'accuracy': 0.9780534351145038, 'precision': 0.9831985065339142, 'recall': 0.9881175734834271, 'f1': 0.9856519026824704, 'roc_auc': 0.9889329724437934}


## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [47]:
# TODO: build and log RandomForestClassifier.

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42
    ))
])

run5 = run_mlflow_experiment(
    "v5_random_forest",
    rf_pipeline,
    X_train, X_test,
    y_train, y_test,
    clean_feature_cols,
    model_params={"model":"rf","n_estimators":300,"max_depth":10},
    tags={"leakage_status":"clean","model_family":"random_forest"}
)

Run completed: v5_random_forest
Run ID: c003222e167542008cf95f5822495131
{'accuracy': 0.9833015267175572, 'precision': 0.986318407960199, 'recall': 0.991869918699187, 'f1': 0.9890863735578422, 'roc_auc': 0.9922801348428282}


## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [48]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# TODO: retrieve MLflow runs for this experiment and create a comparison table.
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

comparison_df = runs_df[[
    "run_id",
    "tags.mlflow.runName",
    "tags.leakage_status",
    "tags.model_family",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1",
    "metrics.roc_auc"
]].sort_values("metrics.f1", ascending=False)

comparison_df

,run_id,tags.mlflow.runName,tags.leakage_status,tags.model_family,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1,metrics.roc_auc
0,c003222e167542008cf95f5822495131,v5_random_forest,clean,random_forest,0.983302,0.986318,0.991870,0.989086,0.992280
1,da423c75e2c94ca0b2a8b104ca1fc952,v4_threshold_0.6,clean,logistic_regression,0.978053,0.983199,0.988118,0.985652,0.988933
3,28e3e8207ca14b78a5be4d9bbd3af30e,v4_threshold_0.4,clean,logistic_regression,0.977099,0.979592,0.990619,0.985075,0.988933
2,69b74317a99d40e5bbdd75792a07801e,v4_threshold_0.5,clean,logistic_regression,0.977099,0.981976,0.988118,0.985037,0.988933
5,6e15e7fea0844db18c3781079ccc8127,v3_balanced_logistic_regression,clean,logistic_regression,0.977099,0.981976,0.988118,0.985037,0.988933
6,77f19294590b4ee6b52e6fbb095e2d61,v2_clean_logistic_regression,clean,logistic_regression,0.975668,0.975430,0.993121,0.984196,0.988169
8,5d70c3a28a54428bade2b0b697e1de96,v0_leaky_logistic_regression,leaky,logistic_regression,0.975668,0.975430,0.993121,0.984196,0.988169
4,62b2d4057a2f40d19294888b8191bc13,v4_threshold_0.3,clean,logistic_regression,0.975668,0.977189,0.991245,0.984166,0.988933
7,d46b3dab31af410ca1a759e10d63caba,v1_dummy_baseline,clean,dummy,0.762882,0.762882,1.000000,0.865494,0.500000


## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [ ]:
# TODO: set BEST_RUN_ID to the selected clean run ID.
BEST_RUN_ID = "c003222e167542008cf95f5822495131"

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("Selected best run:", BEST_RUN_ID)

Selected best run: c003222e167542008cf95f5822495131


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [52]:
# TODO: replace this text.
final_explanation = """
I selected the final clean model based on the combination of F1 score, ROC-AUC,
and a balanced precision/recall tradeoff. The leaky run was intentionally excluded
because it used future availability information, which would not be available at 
prediction time and therefore is unrealistic in a real‑world setting. 
The selected run logged all required MLflow artifacts, parameters, metrics, 
and a full sklearn preprocessing Pipeline, ensuring reproducibility.
"""

print(final_explanation)


I selected the final clean model based on the combination of F1 score, ROC-AUC,
and a balanced precision/recall tradeoff. The leaky run was intentionally excluded
because it used future availability information, which would not be available at 
prediction time and therefore is unrealistic in a real‑world setting. 
The selected run logged all required MLflow artifacts, parameters, metrics, 
and a full sklearn preprocessing Pipeline, ensuring reproducibility.

